# Targeted logZ calibration — N=17/18 (6/20)

这个 Notebook 只重新校准已经被 Step 12 标记异常的 `N=17,18`。它不训练 policy、不执行 optimizer update、不重算其他 N，也不读取 validation/OOS。

### 第 0 格：检查配置与安全开关

本格只构造配置并打印 F/D/E/L、target、64/128 稳定性合同和路径，不加载真实数据。先以 `RUN_REAL_TARGETED_CALIBRATION=False` 运行并检查；确认后改为 `True`。

In [ ]:
import json, sys, torch
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import pandas as pd

root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'factor_gfn').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from factor_gfn.gfn import (
    ExhaustiveRegistry,
    GFNTrainer,
    NoAnchorCalibrationConfig,
    NoAnchorComplexityConfig,
    RealRewardDataPaths,
    RealRewardProvider,
    TrainingConfig,
    build_formal_stage5_no_anchor_6_20_config,
    build_real_reward_data_context,
    load_targeted_calibration_progress,
    policy_state_fingerprint,
    save_targeted_calibration_progress,
    write_targeted_calibration_artifact,
)
from factor_gfn.gfn.diagnostic_support import (
    PhaseTrackingRewardProvider,
    configure_registry_once,
    progress_heartbeat,
    run_targeted_calibration_with_progress,
)

RUN_REAL_TARGETED_CALIBRATION = False
RUN_MODE = 'new'  # 中断后改为 'resume'，不要删除已有目录
DEVICE = 'cuda:0'
RUN_NAME = 'targeted_logz_n17_n18_seed42'
TARGET_NODE_COUNTS = (17, 18)

SOURCE_DIAGNOSTIC_ROOT = root / 'runs' / 'complexity_diagnostic_6_20' / 'manual_diagnostic_6_20_seed42'
SOURCE_REGISTRY = SOURCE_DIAGNOSTIC_ROOT / 'exhaustive_registry.sqlite3'
RUN_ROOT = root / 'runs' / 'targeted_calibration_6_20' / RUN_NAME
PROGRESS_CHECKPOINT = RUN_ROOT / 'latest_targeted_calibration.pt'
RESULT_ARTIFACT = RUN_ROOT / 'targeted_log_z_calibration.json'

training = TrainingConfig(
    batch_size=8,
    learning_rate=1e-4,
    log_z_learning_rate=1e-2,
    max_steps=1,
    model_gradient_clip_norm=5.0,
    log_z_gradient_clip_norm=5.0,
    seed=42,
)
base_config = build_formal_stage5_no_anchor_6_20_config(training=training)
config = replace(
    base_config,
    complexity=NoAnchorComplexityConfig(
        exact_normalizer_node_counts=(1, 2),
        exact_node_retry_budget=3,
    ),
    calibration=NoAnchorCalibrationConfig(
        enabled=True,
        target_node_counts=TARGET_NODE_COUNTS,
        minimum_valid_samples=64,
        maximum_requested_slots_per_N=128,
        comparison_window=16,
        median_absolute_tolerance=0.25,
        iqr_absolute_tolerance=0.50,
    ),
)
strata = config.resolved_strata()
print({
    'run_enabled': RUN_REAL_TARGETED_CALIBRATION,
    'run_mode': RUN_MODE,
    'device': DEVICE,
    'run_root': str(RUN_ROOT),
    'source_diagnostic_root': str(SOURCE_DIAGNOSTIC_ROOT),
    'F': strata.feasible_node_counts,
    'D': strata.discovery_node_counts,
    'E': strata.exact_normalizer_node_counts,
    'L': strata.learned_normalizer_node_counts,
    'targets': TARGET_NODE_COUNTS,
    'retry_budget': config.complexity.exact_node_retry_budget,
    'calibration': asdict(config.calibration),
    'config_fingerprint': config.fingerprint(),
}, flush=True)
print('本 Notebook 只生成初始化信息；policy/optimizer/scheduler训练状态不会被复用。', flush=True)


### 第 1 格：建立或恢复 calibration-only run

本格加载真实 training-only 数据，预计通常需要数分钟；每20秒输出心跳。随后只读打开旧 N=1/2 registry，执行一次等价性证明，并导入除 N=17/18 外的历史 median。`resume` 只恢复 targeted calibration 的观测、独立 scheduler 和 RNG。

In [ ]:
if not RUN_REAL_TARGETED_CALIBRATION:
    raise RuntimeError('安全停止：检查第0格后，将 RUN_REAL_TARGETED_CALIBRATION=True')
if RUN_MODE not in {'new', 'resume'}:
    raise ValueError("RUN_MODE 只能是 'new' 或 'resume'")
if not DEVICE.startswith('cuda:') or not torch.cuda.is_available():
    raise RuntimeError('Targeted calibration 必须显式使用 CUDA，不回落 CPU')
for required in ('diagnostic_summary.json', 'diagnostic_context.json', 'diagnostic_checkpoint.pt', 'exhaustive_registry.sqlite3'):
    if not (SOURCE_DIAGNOSTIC_ROOT / required).is_file():
        raise FileNotFoundError(SOURCE_DIAGNOSTIC_ROOT / required)

device = torch.device(DEVICE)
torch.cuda.set_device(device)
torch.cuda.reset_peak_memory_stats(device)
if RUN_MODE == 'new':
    RUN_ROOT.mkdir(parents=True, exist_ok=False)
elif not RUN_ROOT.is_dir():
    raise FileNotFoundError('resume 要求已有同名 targeted calibration 目录')

print('[provider] loading training-only context; heartbeat every 20s', flush=True)
provider_started = perf_counter()
with progress_heartbeat('targeted provider load', interval_seconds=20.0):
    context = build_real_reward_data_context(paths=RealRewardDataPaths())
    base_provider = RealRewardProvider(context, config.reward)
    provider = PhaseTrackingRewardProvider(
        base_provider,
        audit_path=RUN_ROOT / 'targeted_reward_audit.jsonl',
    )
manifest = provider.manifest()
assert manifest['data_scope'] == 'training_only'
assert manifest['validation_oos_loaded'] is False
assert manifest['industry_neutralization']['enabled']
assert base_provider.reward_config.candidate_industry_neutralization
print(f'[provider] ready in {perf_counter()-provider_started:.1f}s; fingerprint={provider.fingerprint()}', flush=True)

registry = ExhaustiveRegistry(SOURCE_REGISTRY, read_only=True)
trainer = GFNTrainer(config, provider, device=device)
with progress_heartbeat('registry equivalence proof', interval_seconds=20.0):
    configure_registry_once(trainer, registry)
with progress_heartbeat('historical non-target initialization proof', interval_seconds=20.0):
    historical = trainer.initialize_verified_historical_log_z(SOURCE_DIAGNOSTIC_ROOT)
expected_historical = set(trainer.resolved_learned_node_counts) - set(TARGET_NODE_COUNTS)
assert set(historical.learned_node_counts) == expected_historical
assert trainer.step == 0 and trainer.optimizer_step == 0 and not trainer.optimizer.state

if RUN_MODE == 'resume':
    if not PROGRESS_CHECKPOINT.is_file():
        raise FileNotFoundError('resume 未找到 latest_targeted_calibration.pt')
    load_targeted_calibration_progress(PROGRESS_CHECKPOINT, trainer)
    print('[resume] restored calibration-only scheduler/observations/RNG', flush=True)
else:
    save_targeted_calibration_progress(PROGRESS_CHECKPOINT, trainer)
    print('[new] initial calibration-only progress saved', flush=True)

run_context = {
    'schema': 'factor_gfn.targeted_log_z_calibration_run.v1',
    'config_fingerprint': config.fingerprint(),
    'provider_fingerprint': provider.fingerprint(),
    'context_fingerprint': manifest['context_fingerprint'],
    'historical_provenance_fingerprint': historical.provenance_fingerprint,
    'targets': list(TARGET_NODE_COUNTS),
    'policy_state_fingerprint': policy_state_fingerprint(trainer),
    'registry_read_only': registry.read_only,
}
context_path = RUN_ROOT / 'targeted_calibration_context.json'
if RUN_MODE == 'new':
    context_path.write_text(json.dumps(run_context, ensure_ascii=False, indent=2), encoding='utf-8')
elif json.loads(context_path.read_text(encoding='utf-8')) != run_context:
    raise RuntimeError('resume context 与当前语义不一致')
print('[ready]', {
    'requested': trainer.calibration.requested_by_N,
    'valid': trainer.calibration.valid_by_N,
    'sampled_attempts': trainer.calibration.sampled_attempts_by_N,
    'status': trainer.calibration.status,
    'historical_N': historical.learned_node_counts,
}, flush=True)


### 第 2 格：运行 N=17/18 targeted calibration

这是唯一的长计算格。N=17和18由独立 balanced scheduler 公平轮换；每个 slot 最多为同一 N 尝试4条 trajectory（首次+3次额外 retry）。每个 slot 都保存 calibration-only checkpoint，每20个累计 slot输出 requested/valid/attempts、窗口稳定性、耗时和上界 ETA；单个 slot 超过20秒时持续输出心跳。达到64 valid后按16-valid窗口检查，必要时继续，最迟128 requested/N 时 fail-closed。

In [ ]:
calibration_started = perf_counter()
with provider.phase('targeted_calibration'):
    report = run_targeted_calibration_with_progress(
        trainer,
        progress_checkpoint_path=PROGRESS_CHECKPOINT,
        progress_every=20,
    )
calibration_seconds = perf_counter() - calibration_started
assert trainer.calibration.status == 'complete'
assert trainer.step == 0 and trainer.optimizer_step == 0 and not trainer.optimizer.state
artifact = write_targeted_calibration_artifact(RESULT_ARTIFACT, trainer)

rows = []
for node_count in TARGET_NODE_COUNTS:
    stats = report[node_count]
    stable = trainer.calibration.stability_by_N[node_count]
    rows.append({
        **stats,
        'stability_status': stable.status,
        'stability_reason': stable.reason,
        'previous_window_median': stable.previous_window_median,
        'recent_window_median': stable.recent_window_median,
        'median_shift': stable.median_shift,
        'previous_window_iqr': stable.previous_window_iqr,
        'recent_window_iqr': stable.recent_window_iqr,
        'iqr_shift': stable.iqr_shift,
    })
result_frame = pd.DataFrame(rows).set_index('node_count').sort_index()
result_frame.to_csv(RUN_ROOT / 'targeted_log_z_calibration_by_N.csv')
summary = {
    'schema': artifact['schema'],
    'targets': list(TARGET_NODE_COUNTS),
    'calibration_seconds': calibration_seconds,
    'requested_by_N': trainer.calibration.requested_by_N,
    'valid_by_N': trainer.calibration.valid_by_N,
    'sampled_attempts_by_N': trainer.calibration.sampled_attempts_by_N,
    'median_log_z_by_N': artifact['median_log_z_by_N'],
    'stability_by_N': artifact['calibration_stability_by_N'],
    'artifact_fingerprint': artifact['artifact_fingerprint'],
    'training_only': manifest['data_scope'] == 'training_only',
    'validation_oos_not_loaded': manifest['validation_oos_loaded'] is False,
    'policy_frozen': trainer.step == 0 and trainer.optimizer_step == 0 and not trainer.optimizer.state,
    'cuda_peak_memory_bytes': int(torch.cuda.max_memory_allocated(device)),
}
(RUN_ROOT / 'targeted_calibration_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(result_frame)
print('[complete]', {
    'wall_seconds': calibration_seconds,
    'artifact': str(RESULT_ARTIFACT),
    'artifact_fingerprint': artifact['artifact_fingerprint'],
    'requested': trainer.calibration.requested_by_N,
    'valid': trainer.calibration.valid_by_N,
    'sampled_attempts': trainer.calibration.sampled_attempts_by_N,
}, flush=True)


### 第 3 格：用全新 Trainer 验证 artifact 可安全导入

本格不执行 Reward 评价或训练。它重新创建相同初始 policy，重新完成 registry/历史来源证明，再严格导入 N=17/18 新 median；验证其他 N 仍为历史值、模型与 optimizer 未恢复或改变。预计数秒到约1分钟，等价证明超过20秒会输出心跳。

In [ ]:
fresh = GFNTrainer(config, provider, device=device)
with progress_heartbeat('fresh registry equivalence proof', interval_seconds=20.0):
    configure_registry_once(fresh, registry)
with progress_heartbeat('fresh historical initialization proof', interval_seconds=20.0):
    fresh_historical = fresh.initialize_verified_historical_log_z(SOURCE_DIAGNOSTIC_ROOT)
policy_before_import = policy_state_fingerprint(fresh)
targeted_record = fresh.initialize_verified_targeted_log_z(RESULT_ARTIFACT)
policy_after_import = policy_state_fingerprint(fresh)

assert policy_before_import == policy_after_import
assert fresh.step == 0 and fresh.optimizer_step == 0 and not fresh.optimizer.state
assert targeted_record.target_node_counts == TARGET_NODE_COUNTS
assert set(fresh_historical.learned_node_counts) == expected_historical
assert set(targeted_record.median_log_z_by_N) == set(TARGET_NODE_COUNTS)
assert all(bool(fresh.tb_loss.learned_log_z_initialized_mask[n-1]) for n in fresh.resolved_learned_node_counts)

hybrid_rows = []
for n in fresh.resolved_learned_node_counts:
    source = 'targeted_recalibration' if n in TARGET_NODE_COUNTS else 'verified_historical_median'
    expected = (
        targeted_record.median_log_z_by_N[n]
        if n in TARGET_NODE_COUNTS
        else fresh_historical.median_log_z_by_N[n]
    )
    actual = float(fresh.tb_loss.log_z_by_node_count[n-1])
    hybrid_rows.append({'N': n, 'source': source, 'expected_log_z': expected, 'actual_log_z': actual})
    assert actual == torch.tensor(expected, dtype=torch.float32).item()
hybrid_frame = pd.DataFrame(hybrid_rows).set_index('N')
hybrid_frame.to_csv(RUN_ROOT / 'verified_hybrid_initialization_by_N.csv')
display(hybrid_frame)

verification = {
    'schema': 'factor_gfn.targeted_log_z_import_verification.v1',
    'artifact_sha256': targeted_record.source_artifact_sha256,
    'artifact_fingerprint': targeted_record.artifact_fingerprint,
    'historical_provenance_fingerprint': fresh_historical.provenance_fingerprint,
    'policy_state_fingerprint_before_after_equal': policy_before_import == policy_after_import,
    'restored_training_state': targeted_record.restored_training_state,
    'step': fresh.step,
    'optimizer_step': fresh.optimizer_step,
    'optimizer_state_empty': not fresh.optimizer.state,
    'hybrid_initialization_by_N': hybrid_rows,
}
(RUN_ROOT / 'targeted_import_verification.json').write_text(
    json.dumps(verification, ensure_ascii=False, indent=2), encoding='utf-8'
)
registry.close()
print('TARGETED_CALIBRATION_ACCEPTANCE_OK', flush=True)
print('停止：不要在本 Notebook 训练。把本目录结果交给 Codex 检查后，再建立 clipping 5 vs 20 对照。', flush=True)
